In [0]:
# Imports

from pyspark.sql.functions import (
    col, round as spark_round,
    when, lag, avg,
    current_timestamp, date_format,
    min, max, lit, count
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

print("Imports successful")

In [0]:
# Cell 2 — Read all four Silver fact tables

df_gdp = spark.table("workspace.default.silver_fact_gdp")
df_climate = spark.table("workspace.default.silver_fact_climate")
df_food = spark.table("workspace.default.silver_fact_food")
df_population = spark.table("workspace.default.silver_fact_population")

print("Silver table row counts:")
print(f"  GDP:        {df_gdp.count()}")
print(f"  Climate:    {df_climate.count()}")
print(f"  Food:       {df_food.count()}")
print(f"  Population: {df_population.count()}")

In [0]:
# Join all four Silver tables
# We join on country_code AND year — the common key
# that exists across all four Silver fact tables.
#
# Join type: LEFT join from GDP outward.
# Why LEFT and not INNER?
# GDP has the most complete coverage (210 rows, no nulls).
# Food and population are also annual and complete.
# Climate is monthly — we aggregate it to annual first
# so it joins cleanly on country_code + year.
#
# If we used INNER joins across all four, any country+year
# missing from ANY single table would drop the entire row.
# LEFT join keeps all country+year combinations from GDP
# and brings in nulls where other sources have gaps.
# We handle those nulls in the scoring step.
#
# STEP 1: Aggregate climate from monthly to annual
# silver_fact_climate is monthly (2,520 rows).
# GDP, food, and population are annual (210 rows each).
# We must aggregate climate to annual averages first
# so the join key country_code + year works cleanly.

df_climate_annual = (
    df_climate
    .groupBy("country_code", "year")
    .agg(
        spark_round(avg("avg_precipitation_mm"), 2).alias("annual_avg_precipitation_mm"),
        spark_round(avg("avg_temperature_c"),    2).alias("annual_avg_temperature_c"),
    )
)

print(f"Climate aggregated to annual: {df_climate_annual.count()} rows")

# STEP 2: Join all four tables
df_joined = (
    df_gdp
    .select("country_code", "country_name", "country_iso3", "region", "year", "gdp_per_capita_usd")
    .join(
        df_climate_annual,
        on=["country_code", "year"],
        how="left"
    )
    .join(
        df_food.select("country_code", "year", "undernourishment_pct", "stunting_pct", "underweight_pct"),
        on=["country_code", "year"],
        how="left"
    )
    .join(
        df_population.select("country_code", "year", "total_population", "urban_population_pct"),
        on=["country_code", "year"],
        how="left"
    )
)

print(f"Joined table rows: {df_joined.count()}")
print(f"Joined table columns: {len(df_joined.columns)}")
print(f"\nSample — Kenya 2020 (all four domains side by side):")
display(
    df_joined
    .filter((col("country_code") == "KE") & (col("year") == 2020))
)

In [0]:
# Null audit before scoring
# Explicitly document what happens to nulls in the Gold join.
# This is a deliberate engineering decision, not an oversight.
#
# undernourishment_pct: 0 nulls — used as primary food signal
# stunting_pct: 128 nulls — supplementary only, NOT used in scoring
# underweight_pct: 128 nulls — supplementary only, NOT used in scoring
#
# Decision: nulls in stunting/underweight are left as-is in the join
# because the food_score uses only undernourishment_pct which is complete.
# Rows with null stunting/underweight still receive a valid food_score.
# No rows are excluded from the composite index.
# No imputation is applied — imputing food security data with averages
# would misrepresent the severity in countries that don't report.

null_audit = df_joined.select(
    [count(when(col(c).isNull(), 1)).alias(f"nulls_{c}")
     for c in ["gdp_per_capita_usd", "annual_avg_precipitation_mm",
               "annual_avg_temperature_c", "undernourishment_pct",
               "stunting_pct", "underweight_pct", "total_population",
               "urban_population_pct"]]
)

print("NULL AUDIT — pre-scoring joined table:")
display(null_audit)
print("""
DECISION DOCUMENTED:
- undernourishment_pct has 0 nulls → used as sole food security signal in food_score
- stunting_pct has 128 nulls → retained in Gold table for reference, NOT used in composite
- underweight_pct has 128 nulls → same as stunting
- No rows are dropped from the Gold join
- No imputation applied — imputing food security data with averages would
  misrepresent severity in countries that simply do not report these metrics
- All 210 country-year combinations receive a valid vulnerability_index
""")

In [0]:
# Normalise indicators and calculate vulnerability score
#
# THE CORE PROBLEM:
# We cannot add GDP ($1,927) + undernourishment (26.3%) + rainfall (4.37mm)
# directly. They have completely different units and scales.
# Adding them raw would mean GDP dominates the score simply
# because its numbers are largest — not because it matters most.
#
# THE SOLUTION: Min-Max Normalisation
# Convert every indicator to a 0-1 scale where:
#   0 = least vulnerable value across all countries all years
#   1 = most vulnerable value across all countries all years
#
# Formula: normalised = (value - min) / (max - min)
#
# DIRECTION MATTERS:
# Some indicators increase vulnerability as they go UP:
#   undernourishment_pct UP → more vulnerable → normalise directly
#
# Some indicators increase vulnerability as they go DOWN:
#   gdp_per_capita_usd DOWN → more vulnerable → flip: 1 - normalised
#   annual_avg_precipitation_mm DOWN → drought → more vulnerable → flip
#
# After normalisation, every indicator is 0-1.
# We multiply by domain weight and sum to get the final score 0-100.

# STEP 1: Calculate min and max for each indicator across ALL rows
# These become our normalisation bounds.
# We collect them as Python scalars using .agg().collect()

bounds = df_joined.agg(
    min("gdp_per_capita_usd").alias("gdp_min"),
    max("gdp_per_capita_usd").alias("gdp_max"),
    min("annual_avg_precipitation_mm").alias("precip_min"),
    max("annual_avg_precipitation_mm").alias("precip_max"),
    min("annual_avg_temperature_c").alias("temp_min"),
    max("annual_avg_temperature_c").alias("temp_max"),
    min("undernourishment_pct").alias("food_min"),
    max("undernourishment_pct").alias("food_max"),
    min("urban_population_pct").alias("urban_min"),
    max("urban_population_pct").alias("urban_max"),
).collect()[0]

print("Normalisation bounds:")
print(f"  GDP:            ${bounds['gdp_min']:,.0f} → ${bounds['gdp_max']:,.0f}")
print(f"  Precipitation:  {bounds['precip_min']} → {bounds['precip_max']} mm")
print(f"  Temperature:    {bounds['temp_min']} → {bounds['temp_max']} °C")
print(f"  Undernourishment: {bounds['food_min']} → {bounds['food_max']} %")
print(f"  Urban %:        {bounds['urban_min']} → {bounds['urban_max']} %")

In [0]:
# Apply normalisation and calculate domain scores
#
# DOMAIN SCORES (each 0-1 before weighting):
#
# ECONOMIC SCORE (weight 25%)
# Low GDP → high vulnerability → we flip: 1 - normalised_gdp
# Higher economic score = poorer economy = more vulnerable
#
# CLIMATE SCORE (weight 25%)
# Low precipitation → drought risk → flip: 1 - normalised_precip
# High temperature → heat stress → normalise directly
# Climate score = average of drought risk + heat stress
#
# FOOD SCORE (weight 35%)
# High undernourishment → more vulnerable → normalise directly
# We use only undernourishment (complete data, 0 nulls)
# stunting and underweight have 128 nulls — unreliable for scoring
#
# POPULATION PRESSURE SCORE (weight 15%)
# Low urban % → rural populations are more exposed to
# climate shocks and food insecurity → flip: 1 - normalised_urban
#
# FINAL VULNERABILITY INDEX:
# Weighted sum of all four domain scores × 100
# Scale: 0 (least vulnerable) to 100 (most vulnerable)

gdp_min    = bounds["gdp_min"]
gdp_max    = bounds["gdp_max"]
precip_min = bounds["precip_min"]
precip_max = bounds["precip_max"]
temp_min   = bounds["temp_min"]
temp_max   = bounds["temp_max"]
food_min   = bounds["food_min"]
food_max   = bounds["food_max"]
urban_min  = bounds["urban_min"]
urban_max  = bounds["urban_max"]

df_scored = (
    df_joined

    # ── ECONOMIC DOMAIN ──────────────────────────────────
    # Normalise GDP then flip (low GDP = high vulnerability)
    .withColumn(
        "economic_score",
        spark_round(
            lit(1.0) - (col("gdp_per_capita_usd") - lit(gdp_min)) / lit(gdp_max - gdp_min),
            4
        )
    )

    # ── CLIMATE DOMAIN ───────────────────────────────────
    # Drought risk: low precipitation = high vulnerability (flipped)
    .withColumn(
        "drought_risk",
        lit(1.0) - (col("annual_avg_precipitation_mm") - lit(precip_min)) / lit(precip_max - precip_min)
    )
    # Heat stress: high temperature = high vulnerability (direct)
    .withColumn(
        "heat_stress",
        (col("annual_avg_temperature_c") - lit(temp_min)) / lit(temp_max - temp_min)
    )
    # Climate score = average of drought risk and heat stress
    .withColumn(
        "climate_score",
        spark_round((col("drought_risk") + col("heat_stress")) / lit(2.0), 4)
    )

    # ── FOOD SECURITY DOMAIN ─────────────────────────────
    # High undernourishment = high vulnerability (direct)
    .withColumn(
        "food_score",
        spark_round(
            (col("undernourishment_pct") - lit(food_min)) / lit(food_max - food_min),
            4
        )
    )

    # ── POPULATION PRESSURE DOMAIN ───────────────────────
    # Low urban % = more rural = more exposed to climate/food shocks
    .withColumn(
        "population_score",
        spark_round(
            lit(1.0) - (col("urban_population_pct") - lit(urban_min)) / lit(urban_max - urban_min),
            4
        )
    )

    # ── COMPOSITE VULNERABILITY INDEX ────────────────────
    # Weighted sum × 100 to get a 0-100 scale
    # Weights: Economic 25%, Climate 25%, Food 35%, Population 15%
    .withColumn(
        "vulnerability_index",
        spark_round(
            (
                col("economic_score")   * lit(0.25) +
                col("climate_score")    * lit(0.25) +
                col("food_score")       * lit(0.35) +
                col("population_score") * lit(0.15)
            ) * lit(100),
            2
        )
    )
)

print(f"Scored rows: {df_scored.count()}")
print(f"\nSample — Kenya all years:")
display(
    df_scored
    .filter(col("country_code") == "KE")
    .select("country_name", "year", "economic_score", "climate_score",
            "food_score", "population_score", "vulnerability_index")
    .orderBy("year")
)

In [0]:
# Trend detection using window functions
#
# WHAT IS A WINDOW FUNCTION?
# A regular groupBy() collapses multiple rows into one.
# A window function looks ACROSS rows without collapsing them.
# Every row keeps its own identity but gains access to
# information from neighbouring rows.
#
# THE PROBLEM WE ARE SOLVING:
# A vulnerability score of 65 tells us a country is vulnerable.
# But it doesn't tell us if things are getting BETTER or WORSE.
# A country at 65 that was 80 last year is improving.
# A country at 65 that was 50 last year is deteriorating fast.
# Trend direction is more actionable than the score alone.
#
# HOW lag() WORKS:
# lag("vulnerability_index", 1) looks at the PREVIOUS row's value
# within the same window partition.
#
# We define the window as:
#   partitionBy("country_code") — one partition per country
#   orderBy("year") — rows ordered by year within each partition
#
# So for Kenya 2015, lag() returns Kenya 2014's score.
# For Kenya 2010 (first year), lag() returns null — no previous year.
#
# YEAR OVER YEAR CHANGE:
# current_score - previous_score
# Positive = vulnerability increased (getting worse)
# Negative = vulnerability decreased (getting better)
#
# TREND LABEL:
# We use when()/otherwise() — PySpark's if-else —
# to convert the numeric change into a readable label.

# Define the window: per country, ordered by year
country_year_window = Window.partitionBy("country_code").orderBy("year")

df_trend = (
    df_scored

    # lag() fetches the previous year's vulnerability index
    # for the same country
    .withColumn(
        "prev_year_index",
        lag("vulnerability_index", 1).over(country_year_window)
    )

    # Year-over-year change: positive = worsening, negative = improving
    .withColumn(
        "yoy_change",
        spark_round(
            col("vulnerability_index") - col("prev_year_index"),
            2
        )
    )

    # Trend label using when()/otherwise()
    .withColumn(
        "trend",
        when(col("yoy_change") > lit(2.0),  "WORSENING")
        .when(col("yoy_change") < lit(-2.0), "IMPROVING")
        .when(col("yoy_change").isNull(),     "BASELINE")
        .otherwise("STABLE")
    )
)

print(f"Trend rows: {df_trend.count()}")
print(f"\nKenya vulnerability trend 2010-2023:")
display(
    df_trend
    .filter(col("country_code") == "KE")
    .select("country_name", "year", "vulnerability_index", "yoy_change", "trend")
    .orderBy("year")
)

In [0]:
# Alert flags
#
# The vulnerability index tells us the score.
# The trend tells us the direction.
# The alert flag tells us what ACTION to take.
#
# We define three alert levels:
#
# HIGH RISK (score >= 70):
# Country is in the top vulnerability range.
# Immediate attention warranted.
# Historical analysis shows scores above 70 correlate
# with countries that experienced humanitarian crises
# within 1-2 years.
#
# ELEVATED (score >= 55):
# Country is above average vulnerability.
# Monitor closely, especially if trend is WORSENING.
#
# MONITOR (score >= 40):
# Country has moderate vulnerability.
# Standard monitoring cadence.
#
# STABLE (score < 40):
# Country is in lower vulnerability range.
# No immediate concern.
#
# when()/otherwise() chains work like SQL CASE WHEN THEN:
# Spark evaluates conditions top to bottom.
# First condition that is TRUE wins.
# otherwise() is the ELSE clause — catches everything remaining.

df_flagged = (
    df_trend
    .withColumn(
        "alert_level",
        when(col("vulnerability_index") >= lit(70), "HIGH RISK")
        .when(col("vulnerability_index") >= lit(55), "ELEVATED")
        .when(col("vulnerability_index") >= lit(40), "MONITOR")
        .otherwise("STABLE")
    )

    # Add processing timestamp
    .withColumn(
        "scored_at",
        date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")
    )

    # Select final Gold table columns in clean order
    .select(
        "country_code",
        "country_iso3",
        "country_name",
        "region",
        "year",
        "gdp_per_capita_usd",
        "annual_avg_precipitation_mm",
        "annual_avg_temperature_c",
        "undernourishment_pct",
        "total_population",
        "urban_population_pct",
        "economic_score",
        "climate_score",
        "food_score",
        "population_score",
        "vulnerability_index",
        "yoy_change",
        "trend",
        "alert_level",
        "scored_at"
    )
)

print(f"Final Gold rows: {df_flagged.count()}")
print(f"\nAlert level distribution:")
display(
    df_flagged.groupBy("alert_level")
              .count()
              .orderBy("count", ascending=False)
)

In [0]:
# Write to Gold Delta table
# Gold uses overwrite — it is a fully derived table.
# Every run recalculates scores from scratch using
# the latest Silver data. There is no history to preserve
# in Gold itself — history lives in Silver and Bronze.
# If we need to see how scores changed over pipeline runs,
# Delta time travel handles that.

TABLE_NAME = "workspace.default.gold_vulnerability_scores"

(df_flagged.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_NAME)
)

print(f"Gold table written: {TABLE_NAME}")
print(f"Row count: {spark.table(TABLE_NAME).count()}")

In [0]:
# The moment of truth
# This is what everything was built for.
# Top 10 most vulnerable country-year combinations
# across all 15 countries and 14 years.
# These are the situations where economic stress,
# climate shock, food insecurity, and population
# pressure converged most severely.

df_gold = spark.table("workspace.default.gold_vulnerability_scores")

print("TOP 10 MOST VULNERABLE COUNTRY-YEAR COMBINATIONS:")
display(
    df_gold
    .orderBy("vulnerability_index", ascending=False)
    .select(
        "country_name", "year", "region",
        "vulnerability_index", "alert_level", "trend",
        "economic_score", "climate_score",
        "food_score", "population_score"
    )
    .limit(10)
)

In [0]:
# Most recent vulnerability ranking (2023)
# Which countries are most at risk RIGHT NOW?

print("VULNERABILITY RANKING — 2023 (Most Recent Year):")
display(
    df_gold
    .filter(col("year") == 2023)
    .orderBy("vulnerability_index", ascending=False)
    .select(
        "country_name", "region",
        "vulnerability_index", "alert_level", "trend",
        "gdp_per_capita_usd", "undernourishment_pct",
        "annual_avg_precipitation_mm"
    )
)

In [0]:
# Countries on the wrong trajectory
# WORSENING trend means the vulnerability index increased
# by more than 2 points compared to the previous year.
# These are the countries that need immediate attention —
# not because they are necessarily the most vulnerable today
# but because they are moving in the wrong direction.

print("COUNTRIES ON WORSENING TRAJECTORY IN 2023:")
display(
    df_gold
    .filter(
        (col("year") == 2023) &
        (col("trend") == "WORSENING")
    )
    .orderBy("vulnerability_index", ascending=False)
    .select(
        "country_name", "region",
        "vulnerability_index", "yoy_change",
        "alert_level", "economic_score",
        "climate_score", "food_score"
    )
)

In [0]:
# Full project audit

all_tables = [
    ("bronze_world_bank_gdp",        210),
    ("bronze_open_meteo_climate",  76695),
    ("bronze_world_bank_food",       374),
    ("bronze_world_bank_population", 420),
    ("dim_country",                   15),
    ("silver_fact_gdp",              210),
    ("silver_fact_climate",         2520),
    ("silver_fact_food",             210),
    ("silver_fact_population",       210),
    ("gold_vulnerability_scores",    210),
]

print("=" * 55)
print("ECONMATE — COMPLETE PIPELINE AUDIT")
print("=" * 55)

all_passed = True
for table, expected in all_tables:
    try:
        actual = spark.table(f"workspace.default.{table}").count()
        status = "." if actual >= expected - 10 else "!"
        if actual < expected - 10:
            all_passed = False
        print(f"{status}  {table}: {actual} rows")
    except Exception as e:
        print(f"✗  {table}: MISSING — {e}")
        all_passed = False

print("=" * 55)
print(f"Pipeline status: {'ALL LAYERS COMPLETE' if all_passed else 'ISSUES FOUND'}")
print("=" * 55)